### Notebook to demonstrate CLIP workflow

CLIP (Contrastive Language-Image Pre-Training) is a neural network trained on a variety of (image, text) pairs. It can be used for zero-shot image classification, image-text retrieval, and more.

### The workflow in a nutshell

- Train
- Evaluate
- Export
- TAO Inference
- TRT Engine generation using TAO-Deploy
- TRT Inference
- Delete jobs

### Table of contents

1. [FIXME's](#head-1)
1. [Login](#head-2)
1. [Create a cloud workspace](#head-3)
1. [Set dataset URIs](#head-4)
1. [Actions](#head-5)
1. [Train](#head-6)
1. [Evaluate](#head-7)
1. [Export](#head-8)
1. [TAO Inference](#head-9)
1. [TRT Engine generation using TAO-Deploy](#head-10)
1. [TRT Inference](#head-11)
1. [Delete jobs](#head-12)

### Requirements
Please find the server requirements [here](https://docs.nvidia.com/tao/tao-toolkit/text/tao_toolkit_api/api_setup.html#)

In [ ]:
# SKIP if already installed.
# pip3 install nvidia-tao-client

In [ ]:
%%bash
tao --version

This notebook uses **Bash** (`%%bash` cells). State between cells is kept in `.tao_nb_state` in this directory. Ensure `tao login` has been run and `jq`/`yq` are installed.

In [ ]:
%%bash
touch .tao_nb_state
source .tao_nb_state
which jq >/dev/null 2>&1 || { echo "Installing jq..."; sudo apt-get install -y jq 2>/dev/null || brew install jq; }
which yq >/dev/null 2>&1 || { echo "Installing yq..."; sudo wget -qO /usr/local/bin/yq https://github.com/mikefarah/yq/releases/latest/download/yq_linux_amd64 && sudo chmod +x /usr/local/bin/yq; }

In [ ]:
%%bash
source .tao_nb_state
echo "State: MODEL_NAME=${MODEL_NAME:-not set} WORKSPACE_ID=${WORKSPACE_ID:-not set}"

### FIXME's <a class="anchor" id="head-1"></a>

1. Set `NGC_KEY` (FIXME 5)
1. Set cloud workspace details (FIXME 7)
1. Set dataset URIs (FIXME 8)
1. Set HuggingFace token (FIXME 9)

In [ ]:
%%bash
source .tao_nb_state
export MODEL_NAME="${TAO_MODEL_NAME:-clip}"
echo "export MODEL_NAME=$MODEL_NAME" >> .tao_nb_state
echo "MODEL_NAME=$MODEL_NAME"

#### Set API service host information

In [ ]:
%%bash
source .tao_nb_state
export NODE_ADDRESS="${NODE_ADDRESS:-localhost}"
export NODE_PORT="${NODE_PORT:-8090}"
export TAO_BASE_URL="http://$NODE_ADDRESS:$NODE_PORT"
echo "export NODE_ADDRESS=$NODE_ADDRESS" >> .tao_nb_state
echo "export NODE_PORT=$NODE_PORT" >> .tao_nb_state
echo "export TAO_BASE_URL=$TAO_BASE_URL" >> .tao_nb_state
echo "TAO_BASE_URL=$TAO_BASE_URL"

#### Set NGC key and org

In [ ]:
%%bash
source .tao_nb_state
# FIXME 5: Set your NGC API key
export NGC_KEY="${NGC_KEY:-your_ngc_key}"
echo "export NGC_KEY=$NGC_KEY" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
export NGC_ORG="${TAO_ORG:-${NGC_ORG:-nvstaging}}"
echo "export NGC_ORG=$NGC_ORG" >> .tao_nb_state
echo "NGC_ORG=$NGC_ORG"

### Login <a class="anchor" id="head-2"></a>

In [ ]:
%%bash
source .tao_nb_state
tao login --ngc-org-name "${NGC_ORG:-nvstaging}" --ngc-key "${NGC_KEY:-your_ngc_key}" --enable-telemetry

### Create a cloud workspace <a class="anchor" id="head-3"></a>

In [ ]:
%%bash
source .tao_nb_state
# FIXME 7: Set cloud workspace details
export TAO_WORKSPACE_NAME="${TAO_WORKSPACE_NAME:-Test Workspace}"
export TAO_WORKSPACE_CLOUD_TYPE="${TAO_WORKSPACE_CLOUD_TYPE:-aws}"
export TAO_WORKSPACE_CLOUD_REGION="${TAO_WORKSPACE_CLOUD_REGION:-us-west-1}"
export TAO_WORKSPACE_CLOUD_BUCKET_NAME="${TAO_WORKSPACE_CLOUD_BUCKET_NAME:-bucket_name}"
export TAO_WORKSPACE_CLOUD_ACCESS_KEY="${TAO_WORKSPACE_CLOUD_ACCESS_KEY:-access_key}"
export TAO_WORKSPACE_CLOUD_SECRET_KEY="${TAO_WORKSPACE_CLOUD_SECRET_KEY:-secret_key}"
echo "export TAO_WORKSPACE_CLOUD_TYPE=$TAO_WORKSPACE_CLOUD_TYPE" >> .tao_nb_state
echo "export TAO_WORKSPACE_CLOUD_REGION=$TAO_WORKSPACE_CLOUD_REGION" >> .tao_nb_state
echo "export TAO_WORKSPACE_CLOUD_BUCKET_NAME=$TAO_WORKSPACE_CLOUD_BUCKET_NAME" >> .tao_nb_state
echo "export TAO_WORKSPACE_CLOUD_ACCESS_KEY=$TAO_WORKSPACE_CLOUD_ACCESS_KEY" >> .tao_nb_state
echo "export TAO_WORKSPACE_CLOUD_SECRET_KEY=$TAO_WORKSPACE_CLOUD_SECRET_KEY" >> .tao_nb_state
echo "export TAO_WORKSPACE_NAME=$TAO_WORKSPACE_NAME" >> .tao_nb_state

WORKSPACE_RESPONSE=$(tao $MODEL_NAME create-workspace-$TAO_WORKSPACE_CLOUD_TYPE \
  --cloud-region "$TAO_WORKSPACE_CLOUD_REGION" \
  --cloud-bucket-name "$TAO_WORKSPACE_CLOUD_BUCKET_NAME" \
  --access-key "$TAO_WORKSPACE_CLOUD_ACCESS_KEY" --secret-key "$TAO_WORKSPACE_CLOUD_SECRET_KEY" \
  --name "$TAO_WORKSPACE_NAME" --output json)
WORKSPACE_ID=$(echo "$WORKSPACE_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$WORKSPACE_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "WORKSPACE_ID=$WORKSPACE_ID"
echo "export WORKSPACE_ID=$WORKSPACE_ID" >> .tao_nb_state

### Set dataset URIs <a class="anchor" id="head-4"></a>

In [ ]:
%%bash
source .tao_nb_state
# FIXME 8: Set dataset URIs relative to the cloud bucket
export TRAIN_DATASET_URI="${TAO_TRAIN_DATASET_URI:-/data/clip_train}"
export EVAL_DATASET_URI="${TAO_EVAL_DATASET_URI:-/data/clip_val}"
echo "export TRAIN_DATASET_URI=$TRAIN_DATASET_URI" >> .tao_nb_state
echo "export EVAL_DATASET_URI=$EVAL_DATASET_URI" >> .tao_nb_state
echo "TRAIN_DATASET_URI=$TRAIN_DATASET_URI"
echo "EVAL_DATASET_URI=$EVAL_DATASET_URI"

In [ ]:
%%bash
source .tao_nb_state
# FIXME 9: HuggingFace token (required to avoid rate limiting when downloading CLIP model weights)
export HF_TOKEN="${HF_TOKEN:-your_hf_token}"
echo "export HF_TOKEN=$HF_TOKEN" >> .tao_nb_state
echo "HF_TOKEN configured: $([ \"$HF_TOKEN\" != \"your_hf_token\" ] && echo yes || echo 'NOT SET - please fix')"

In [ ]:
%%bash
source .tao_nb_state
export ENCODE_KEY="${ENCODE_KEY:-nvidia_tao}"
export DS_FORMAT="default"
echo "export ENCODE_KEY=$ENCODE_KEY" >> .tao_nb_state
echo "export DS_FORMAT=$DS_FORMAT" >> .tao_nb_state

### Actions <a class="anchor" id="head-5"></a>

### Train <a class="anchor" id="head-6"></a>

In [ ]:
%%bash
source .tao_nb_state
tao $MODEL_NAME get-job-schema --defaults-only --action train --output @train_spec.yaml
head -80 train_spec.yaml

In [ ]:
%%bash
source .tao_nb_state
cp -f train_spec.yaml train_spec_filled.yaml
# Set number of epochs
yq -i '.train.num_epochs = 1' train_spec_filled.yaml

In [ ]:
%%bash
source .tao_nb_state
JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action train --name ${MODEL_NAME}_training_job \
  --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" \
  --train-dataset-uri "$TRAIN_DATASET_URI" --eval-dataset-uri "$EVAL_DATASET_URI" \
  --dataset-format "$DS_FORMAT" \
  --env HF_TOKEN=$HF_TOKEN \
  --specs @train_spec_filled.yaml --output json)
JOB_ID_TRAIN=$(echo "$JOB_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$JOB_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "JOB_ID_TRAIN=$JOB_ID_TRAIN"
echo "export JOB_ID_TRAIN=$JOB_ID_TRAIN" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
JOB_ID="$JOB_ID_TRAIN"
MAX_WAIT_RETRIES=10
wait_retries=0
while true; do
  STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
  if [ -z "$STATUS" ]; then
    wait_retries=$((wait_retries + 1))
    echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
    if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
      echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
      exit 1
    fi
    sleep 5
    continue
  fi
  wait_retries=0
  echo "Status: $STATUS"
  tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
  sleep 15
done
if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi

### Evaluate <a class="anchor" id="head-7"></a>

In [ ]:
%%bash
source .tao_nb_state
tao $MODEL_NAME get-job-schema --defaults-only --action evaluate --output @eval_spec.yaml
cp -f eval_spec.yaml eval_spec_filled.yaml

In [ ]:
%%bash
# Edit eval_spec_filled.yaml if needed (yq/sed)

In [ ]:
%%bash
source .tao_nb_state
JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action evaluate --name ${MODEL_NAME}_eval \
  --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" --parent-job-id "$JOB_ID_TRAIN" \
  --eval-dataset-uri "$EVAL_DATASET_URI" \
  --dataset-format "$DS_FORMAT" \
  --env HF_TOKEN=$HF_TOKEN \
  --specs @eval_spec_filled.yaml --output json)
JOB_ID_EVAL=$(echo "$JOB_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$JOB_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "JOB_ID_EVAL=$JOB_ID_EVAL"
echo "export JOB_ID_EVAL=$JOB_ID_EVAL" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
JOB_ID="$JOB_ID_EVAL"
MAX_WAIT_RETRIES=10
wait_retries=0
while true; do
  STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
  if [ -z "$STATUS" ]; then
    wait_retries=$((wait_retries + 1))
    echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
    if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
      echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
      exit 1
    fi
    sleep 5
    continue
  fi
  wait_retries=0
  echo "Status: $STATUS"
  tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
  sleep 15
done
if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi

### Export <a class="anchor" id="head-8"></a>

In [ ]:
%%bash
source .tao_nb_state
tao $MODEL_NAME get-job-schema --defaults-only --action export --output @export_spec.yaml
cp -f export_spec.yaml export_spec_filled.yaml

In [ ]:
%%bash
# Edit export_spec_filled.yaml if needed

In [ ]:
%%bash
source .tao_nb_state
JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action export --name ${MODEL_NAME}_export \
  --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" --parent-job-id "$JOB_ID_TRAIN" \
  --env HF_TOKEN=$HF_TOKEN \
  --specs @export_spec_filled.yaml --output json)
JOB_ID_EXPORT=$(echo "$JOB_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$JOB_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "JOB_ID_EXPORT=$JOB_ID_EXPORT"
echo "export JOB_ID_EXPORT=$JOB_ID_EXPORT" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
JOB_ID="$JOB_ID_EXPORT"
MAX_WAIT_RETRIES=10
wait_retries=0
while true; do
  STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
  if [ -z "$STATUS" ]; then
    wait_retries=$((wait_retries + 1))
    echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
    if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
      echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
      exit 1
    fi
    sleep 5
    continue
  fi
  wait_retries=0
  echo "Status: $STATUS"
  tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
  sleep 15
done
if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi

### TAO Inference <a class="anchor" id="head-9"></a>

In [ ]:
%%bash
source .tao_nb_state
tao $MODEL_NAME get-job-schema --defaults-only --action inference --output @tao_inference_spec.yaml
cp -f tao_inference_spec.yaml tao_inference_spec_filled.yaml

In [ ]:
%%bash
# Edit tao_inference_spec_filled.yaml if needed

In [ ]:
%%bash
source .tao_nb_state
JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action inference --name ${MODEL_NAME}_tao_inference \
  --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" --parent-job-id "$JOB_ID_TRAIN" \
  --train-dataset-uri "$TRAIN_DATASET_URI" --eval-dataset-uri "$EVAL_DATASET_URI" \
  --inference-dataset-uri "$EVAL_DATASET_URI" \
  --dataset-format "$DS_FORMAT" \
  --env HF_TOKEN=$HF_TOKEN \
  --specs @tao_inference_spec_filled.yaml --output json)
JOB_ID_TAO_INF=$(echo "$JOB_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$JOB_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "JOB_ID_TAO_INF=$JOB_ID_TAO_INF"
echo "export JOB_ID_TAO_INF=$JOB_ID_TAO_INF" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
JOB_ID="$JOB_ID_TAO_INF"
MAX_WAIT_RETRIES=10
wait_retries=0
while true; do
  STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
  if [ -z "$STATUS" ]; then
    wait_retries=$((wait_retries + 1))
    echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
    if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
      echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
      exit 1
    fi
    sleep 5
    continue
  fi
  wait_retries=0
  echo "Status: $STATUS"
  tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
  sleep 15
done
if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi

### TRT Engine generation using TAO-Deploy <a class="anchor" id="head-10"></a>

- Here, we use the exported model to generate a TRT engine

In [ ]:
%%bash
source .tao_nb_state
tao $MODEL_NAME get-job-schema --defaults-only --action gen_trt_engine --output @gen_trt_engine_spec.yaml
cp -f gen_trt_engine_spec.yaml gen_trt_engine_spec_filled.yaml

In [ ]:
%%bash
# Edit gen_trt_engine_spec_filled.yaml if needed

In [ ]:
%%bash
source .tao_nb_state
JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action gen_trt_engine --name ${MODEL_NAME}_gen_trt_engine \
  --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" --parent-job-id "$JOB_ID_EXPORT" \
  --env HF_TOKEN=$HF_TOKEN \
  --specs @gen_trt_engine_spec_filled.yaml --output json)
JOB_ID_TRT=$(echo "$JOB_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$JOB_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "JOB_ID_TRT=$JOB_ID_TRT"
echo "export JOB_ID_TRT=$JOB_ID_TRT" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
JOB_ID="$JOB_ID_TRT"
MAX_WAIT_RETRIES=10
wait_retries=0
while true; do
  STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
  if [ -z "$STATUS" ]; then
    wait_retries=$((wait_retries + 1))
    echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
    if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
      echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
      exit 1
    fi
    sleep 5
    continue
  fi
  wait_retries=0
  echo "Status: $STATUS"
  tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
  sleep 15
done
if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi

### TRT Inference <a class="anchor" id="head-11"></a>

In [ ]:
%%bash
source .tao_nb_state
tao $MODEL_NAME get-job-schema --defaults-only --action inference --output @trt_inference_spec.yaml
cp -f trt_inference_spec.yaml trt_inference_spec_filled.yaml

In [ ]:
%%bash
# Edit trt_inference_spec_filled.yaml if needed

In [ ]:
%%bash
source .tao_nb_state
JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action inference --name ${MODEL_NAME}_trt_inference \
  --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" --parent-job-id "$JOB_ID_TRT" \
  --inference-dataset-uri "$EVAL_DATASET_URI" \
  --env HF_TOKEN=$HF_TOKEN \
  --specs @trt_inference_spec_filled.yaml --output json)
JOB_ID_TRT_INF=$(echo "$JOB_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$JOB_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "JOB_ID_TRT_INF=$JOB_ID_TRT_INF"
echo "export JOB_ID_TRT_INF=$JOB_ID_TRT_INF" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
JOB_ID="$JOB_ID_TRT_INF"
MAX_WAIT_RETRIES=10
wait_retries=0
while true; do
  STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
  if [ -z "$STATUS" ]; then
    wait_retries=$((wait_retries + 1))
    echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
    if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
      echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
      exit 1
    fi
    sleep 5
    continue
  fi
  wait_retries=0
  echo "Status: $STATUS"
  tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
  sleep 15
done
if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi

### Delete jobs <a class="anchor" id="head-12"></a>

In [ ]:
%%bash
source .tao_nb_state
for j in $JOB_ID_TRAIN $JOB_ID_EVAL $JOB_ID_EXPORT $JOB_ID_TAO_INF $JOB_ID_TRT $JOB_ID_TRT_INF; do
  [ -n "$j" ] && tao $MODEL_NAME delete-job --job-id "$j" --confirm
done